<a href="https://colab.research.google.com/github/juandiego-guerrero/Problem-Set-3-Econ_3916_PS3.ipynb/blob/main/Problem_Set_3_Econ_3916_PS3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS3 — The Number Nobody Can Reproduce

**ECON 3916 · Applied Data Analytics** · Topics 9-11 · 100 points · your role: Analyst, Business Intelligence

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. Write the SQL yourself; that is the skill being assessed and it is the one interviewers test. AI is authorised and required in Phase 4.

## Setup

In [102]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Reproduce Both Numbers (25 pts)

### Step 1.1: Build the warehouse (8 pts)

In [103]:
!pip install -q duckdb                      # the brief's setup step 2 (quick if Colab already has it)
import duckdb
import statsmodels.formula.api as smf       # for Phase 3

con = duckdb.connect()                       # your warehouse: register customers and orders here

# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_CUSTOMERS = 3_000          # at least 3,000 customers ...
MIN_WEST = 2_000             # ... and at least 2,000 of them in the West
NB_N, NB_P = 0.8, 0.12       # n_orders = 1 + NegativeBinomial(n=0.8, p=0.12): mean near 7, long right tail
REV_PER_ORDER = 60           # revenue = 60 * n_orders + Normal(400, 150), floored at 50
REV_MEAN, REV_SD, REV_FLOOR = 400, 150, 50

In [104]:
# --- Step 1.1: build the warehouse --------------------------------
# One row per customer in customers, one row per order in orders, linked by customer_id:
customer_id = np.arange(1, N_CUSTOMERS + 1)

# Region: exactly MIN_WEST customers in the West (a random draw could fall short of 2,000),
# the rest split across three other regions, then shuffled so West is not all at the top
others = rng.choice(["East", "Central", "South"], size=N_CUSTOMERS - MIN_WEST)
region = rng.permutation(np.concatenate([np.repeat("West", MIN_WEST), others]))

# Orders per customer: 1 + NegativeBinomial(n=0.8, p=0.12) -> mean near 7, long right tail.
# The 1 + means every customer has ordered at least once
n_orders = 1 + rng.negative_binomial(NB_N, NB_P, size=N_CUSTOMERS)

# Revenue is DRIVEN BY order count. That correlation is what the join move the average;
# np.maximum floors it at 50, as np.maximum(0, ...) floors fines at 0 in the warm-up
revenue = np.maximum(
    REV_PER_ORDER * n_orders + rng.normal(REV_MEAN, REV_SD, N_CUSTOMERS),
    REV_FLOOR,
).round(2)

# Signup dates spread over three years (the brief asks for the column; Phase 1 does not use it)
signup_date = pd.Timestamp("2022-01-01") + pd.to_timedelta(
    rng.integers(0, 3 * 365, N_CUSTOMERS), unit="D"
)

customers_df = pd.DataFrame({
    "customer_id": customer_id,
    "region": region,
    "revenue": revenue,
    "signup_date": signup_date,
})

# np.repeat writes each customer_id once per order
orders_df = pd.DataFrame({"customer_id": np.repeat(customer_id, n_orders)})
orders_df.insert(0, "order_id", np.arange(1, len(orders_df) + 1))

# Register with DuckDB and copy into real tables, so Step 2.1 can UPDATE region later
con.register("customers_df", customers_df)
con.register("orders_df", orders_df)
con.execute("CREATE OR REPLACE TABLE customers AS SELECT * FROM customers_df")
con.execute("CREATE OR REPLACE TABLE orders AS SELECT * FROM orders_df")

# --- Confirm the warehouse, and check each table's grain --------
n_cust, n_cust_ids, n_west = con.execute("""
      SELECT COUNT(*), COUNT(DISTINCT customer_id),
             COUNT(*) FILTER (WHERE region = 'West')
      FROM customers
""").fetchone()
n_ord, n_ord_ids = con.execute(
    "SELECT COUNT(*), COUNT(DISTINCT customer_id) FROM orders"
).fetchone()

assert n_cust >= 3_000 and n_west >= 2_000, "brief: at least 3,000 customers, 2,000 in the West"
assert n_cust == n_cust_ids, "customers should have one row per customer"
assert n_ord == n_orders.sum(), "orders should hold one row per draw"
assert n_ord_ids == n_cust, "every customer places at least one order (n_orders = 1 + ...)"

r = np.corrcoef(n_orders, revenue)[0,1]
# :, puts commas between thousands; .2f keeps two decimals
print(f"customers: {n_cust:,} rows for {n_cust_ids:,} customers ({n_west:,} West) -> one row per customer")
print(f"orders: {n_ord:,} rows for {n_ord_ids:,} customers -> several rows per customer")
print(f"orders per customer: mean {n_orders.mean():.2f}, max {n_orders.max()} | "
      f"corr(revenue, n_orders) = {r:.2f}")
print(con.sql("SELECT * FROM customers ORDER BY customer_id LIMIT 5"))

customers: 3,000 rows for 3,000 customers (2,000 West) -> one row per customer
orders: 20,570 rows for 3,000 customers -> several rows per customer
orders per customer: mean 6.86, max 57 | corr(revenue, n_orders) = 0.94
┌─────────────┬─────────┬─────────┬─────────────────────┐
│ customer_id │ region  │ revenue │     signup_date     │
│    int64    │ varchar │ double  │    timestamp_ns     │
├─────────────┼─────────┼─────────┼─────────────────────┤
│           1 │ West    │  843.53 │ 2024-05-01 00:00:00 │
│           2 │ West    │  546.37 │ 2023-06-10 00:00:00 │
│           3 │ West    │  2233.2 │ 2022-03-27 00:00:00 │
│           4 │ West    │ 1171.16 │ 2022-03-13 00:00:00 │
│           5 │ West    │  687.04 │ 2023-07-22 00:00:00 │
└─────────────┴─────────┴─────────┴─────────────────────┘



### Step 1.2: Write both queries (7 pts)

In [105]:
# --- Step 1.2: the two queries --------------------------------
# Same column, same customers, same filter. The only difference is the join to orders.

# Low number: one row per customer, so each customer counts once
LOW_SQL = """
    SELECT AVG(REVENUE)                         AS avg_revenue,
        COUNT(*)                                AS n_rows,
        COUNT(DISTINCT customer_id)             AS n_customers,
        STDDEV_SAMP(revenue) / SQRT(COUNT(*))   AS se
    FROM customers
    WHERE region = 'West'
"""

# High number: the join repeats each customer once per order they placed,
# so AVG runs over orders, not customers
HIGH_SQL = """
    SELECT AVG(REVENUE)                         AS avg_revenue,
        COUNT(*)                                AS n_rows,
        COUNT(DISTINCT c.customer_id)           AS n_customers,
        STDDEV_SAMP(c.revenue) / SQRT(COUNT(*)) AS se
    FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    WHERE c.region = 'West'
"""

low, n_low, cust_low, se_low = con.execute(LOW_SQL).fetchone()
high, n_high, cust_high, se_high = con.execute(HIGH_SQL).fetchone()
ratio = high / low

# Why the join moves the average: it weights each customer's revenue by their order count.
# Collapse orders to one row per customer FIRST, then compute that weighted average directly.
WEIGHTED_SQL = """
    WITH order_counts AS (
        SELECT customer_id, COUNT(*) AS n_orders
        FROM orders
        GROUP BY customer_id
    )
    SELECT SUM(c.revenue * k.n_orders) / SUM(k.n_orders)
    FROM customers c
    JOIN order_counts k ON k.customer_id = c.customer_id
    WHERE c.region = 'West'
  """

weighted = con.execute(WEIGHTED_SQL).fetchone()[0]

# --- Report --------------------------------------
# <30 pads the label to 30 characters; >12, .2f right-aligns with commas and 2 decimals
print(f"{'West region':<30}{'avg revenue':>12}{'rows':>9}{'customers':>11}{'SE':>8}")
print("-" * 70)
print(f"{'customers only (un-joined)':<30}{low:>12,.2f}{n_low:>9,}{cust_low:>11,}{se_low:>8.2f}")
print(f"{'orders joined (joined)':<30}{high:>12,.2f}{n_high:>9,}{cust_high:>11,}{se_high:>8.2f}")
print(f"\nratiojoined / un-joined = {ratio: .3f}")
print(f"rows per West customer after the join: {n_high / cust_high: .2f}")
print(f"order-weighted average of customer revenue: {weighted:,.2f} (= the joined number)")

# --- Verification checkpoint (brief: do not proceed until these pass) ----
assert 780 <= low <= 845, f"un-joined average {low:,.2f} is outside $780 to $845: revisit Step 1.1"
assert 1160 <= high <= 1320, f"joined average {high:,.2f} is outside $1,160 to $1,320: revisit Step 1.2"
assert 1.45 <= ratio <= 1.60, f"ratio {ratio:.3f} is outside 1.45 to 1.60: revenue may not be tied to order count"
assert cust_low == cust_high, "both queries should cover the same West customers"
assert abs(weighted - high) < 0.01, "the joined average should equal the order-weighted average"
print("\nCheckpoint passed.")


West region                    avg revenue     rows  customers      SE
----------------------------------------------------------------------
customers only (un-joined)          823.93    2,000      2,000   10.07
orders joined (joined)            1,256.76   14,004      2,000    5.43

ratiojoined / un-joined =  1.525
rows per West customer after the join:  7.00
order-weighted average of customer revenue: 1,256.76 (= the joined number)

Checkpoint passed.


> **🛑 Verification checkpoint.** On West-region customers you should get about
> **$812** from the un-joined query (accept $780 to $845) and about **$1,240**
> from the joined one (accept $1,160 to $1,320), a ratio between **1.45 and
> 1.60**. If the two numbers are close together, your revenue is not correlated
> with order count, there is no fan-out to find, and the rest of the problem set
> has nothing to work on — go back to Step 1.1. **Do not proceed until it
> matches.**

Both queries average revenue for the same 2,000 West customers. The un-joined query runs over 2,000 rows, one per customer, and returns 823.93. The joined query runs over 14,004, because joining orders repeats each customer once per order (7 per customer), and returns 1,256.76. The ratio is 1.525, so the checkpoint is met.

The metric is revenue per customer, so the unit of observation must be the customer, and revenue is recorded once per customer. The join changes the unit to the order, so a customer with 40 orders counts forty times. Since revenue rises with order count, the repeated rows are the high-revenue customers, and the average is pulled up to an order-weighted figure (weighting by order count gives exactly 1,256.76). The dashboard's 1,240 is that order-weighted number; Finance's 812 is true average per customer.



### Step 1.3: The rule (10 pts)

In [106]:
# --- Step 1.3: the rule ----------------------------------
# A per-customer average needs one row per customer. Before averaging over a join,
# compare its rows with the rows before the join: if the count changed, or customers
# repeat, the average is no longer per customer.

# The rows behind each Step 1.2 number, before AVG is applied
PLAIN_ROWS = "SELECT customer_id, revenue FROM customers WHERE region = 'West'"
JOINED_ROWS = """
    SELECT c.customer_id, c.revenue
    FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    WHERE c.region = 'West'
"""

def fanout_check(con, row_sql, base_sql=PLAIN_ROWS, key="customer_id"):
  """Return a warning string if the rows repeat 'key' or differ in count from base_sql, else None."""
  n_rows, n_keys = con.execute(
      f"SELECT COUNT(*), COUNT(DISTINCT {key}) FROM ({row_sql})"
  ).fetchone()
  n_base = con.execute(f"SELECT COUNT(*) FROM ({base_sql})").fetchone()[0]
  if n_rows != n_keys or n_rows != n_base:
    return (f"ROW COUNT CHANGED: {n_base:,} rows before the join, {n_rows:,} after, "
            f"for {n_keys:,} distinct {key}")
  return None

warn_joined = fanout_check(con, JOINED_ROWS)
warn_plain = fanout_check(con, PLAIN_ROWS)
print("joined (dashboard):", warn_joined)
print ("plain (Finance): ", warn_plain)

assert isinstance(warn_joined, str), "the check should fire on the joined query"
assert warn_plain is None, "the check should stay silent on the un-joined query"

joined (dashboard): ROW COUNT CHANGED: 2,000 rows before the join, 14,004 after, for 2,000 distinct customer_id
plain (Finance):  None


The rule: before averaging over a join, check that the join did not change the row count and that each customer still appears once.

fanout_check compares the rows after a join with the rows before it, and counts distinct customers. On the dashboard's query it fires: the same 2,000 West customers become 14,004 rows, because each customer is repeated once per order. On Finance's query it stays silent: 2,000 rows for 2,000 customers.

A changed row count means the join changed the unit of observation. Extra rows weight the average by order count, and missing rows drop customers. Either way, the number isno longer an average per customer. Because the check takes any query and key, it can run on every dashboard metric before it reaches a slide. Here it would have flagged the 1,240 and pointed to the join as the cause.

## Phase 2: The Rows That Disappear (30 pts)

### Step 2.1: Introduce the unknown (5 pts)

In [107]:
# --- Step 2.1: introduce the unknown ---------------
# About 15% of customers were onboarded before region was a mandatory field,
# so their region is unknown (NULL)
NULL_SHARE = 0.15
n_unknown = int(round(NULL_SHARE * N_CUSTOMERS))          # 450 customers

# Pick them at random, without replacement, so no customer is chosen twice
unknown_ids = rng.choice(customers_df["customer_id"].to_numpy(), size=n_unknown, replace=False)

# Set their region to NULL in the DuckDB table (customers_df keeps the true regions)
con.register("unknown_ids", pd.DataFrame({"customer_id": unknown_ids}))
con.execute("""
    UPDATE customers
    SET region = NULL
    WHERE customer_id IN (SELECT customer_id FROM unknown_ids)
""")

# Confirm: exactly n_unknown NULLs, and no customer lost
n_total, n_null = con.execute("""
      SELECT COUNT(*), COUNT(*) FILTER (WHERE region IS NULL)
      FROM customers
""").fetchone()
assert n_total == N_CUSTOMERS, "the UPDATE should not add or remove customers"
assert n_null == n_unknown, "expected exactly 15% NULL regions: restart and run all if this cell ran twice"

print(f"{n_null:,} of {n_total:,} customers ({n_null / n_total:.1%}) now have region = NULL")
print(con.sql("""
    SELECT region, COUNT(*) AS customers
    FROM customers
    GROUP BY region
    ORDER BY region NULLS LAST
"""))

450 of 3,000 customers (15.0%) now have region = NULL
┌─────────┬───────────┐
│ region  │ customers │
│ varchar │   int64   │
├─────────┼───────────┤
│ Central │       284 │
│ East    │       284 │
│ South   │       280 │
│ West    │      1702 │
│ NULL    │       450 │
└─────────┴───────────┘



### Step 2.2: Lose them, then find them (15 pts)

In [108]:
# --- Step 2.2: lose them, then find them ------------
# The analyst's filter: "every customer outside the West"
NE_SQL = """
    SELECT COUNT (*), AVG(revenue)
    FROM customers
    WHERE region != 'West'
"""

# What the analyst meant: everyone who is not known to be West, unknowns included.
# IS DISTINCT FROM treats NULL as a value that differs from 'West'
MEANT_SQL = """
    SELECT COUNT (*), AVG(revenue)
    FROM customers
    WHERE region IS DISTINCT FROM 'West'
"""
n_ne, avg_ne = con.execute(NE_SQL).fetchone()
n_meant, avg_meant = con.execute(MEANT_SQL).fetchone()

# The same fix in older SQL: add the NULLs back explicitly
n_or_null = con.execute(
    "SELECT COUNT(*) FROM customers WHERE region != 'West' OR region IS NULL"
).fetchone()[0]

n_west, n_null = con.execute("""
      SELECT COUNT(*) FILTER (WHERE region = 'West'),
             COUNT(*) FILTER (WHERE region IS NULL)
      FROM customers
""").fetchone()

# The mechanism: for a NULL region, != 'West' is neither true nor false but NULL,
# and WHERE keeps only rows where the condition is TRUE
null_ne = con.execute("SELECT NULL != 'West'").fetchone()[0]

print(f"WHERE region != 'West':    {n_ne:,} rows | avg revenue {avg_ne:,.2f}")
print(f"WHERE region IS DISTINCT FROM 'West': {n_meant:,} rows | avg revenue {avg_meant:,.2f}")
print(f"difference: {n_meant - n_ne:,} rows = the {n_null:,} customers with NULL region")
print(f"NULL != 'West' evaluates to: {null_ne}")
print(f"West ({n_west:,}) + 'not West' ({n_ne:,}) = {n_west + n_ne:,} of {N_CUSTOMERS} customers "
      f"-> {N_CUSTOMERS - n_west - n_ne:,} fall out of both sides")

assert n_meant - n_ne == n_null, "the gap between the two queries should be exactly the NULL regions"
assert n_or_null == n_meant, "IS DISTINCT FROM and '!= OR IS NULL' should return the same rows"
assert n_west + n_meant == N_CUSTOMERS, "West plus 'not West' should cover every customer"
assert null_ne is None, "NULL compared with anything is NULL (unknown)"


WHERE region != 'West':    848 rows | avg revenue 797.02
WHERE region IS DISTINCT FROM 'West': 1,298 rows | avg revenue 796.26
difference: 450 rows = the 450 customers with NULL region
NULL != 'West' evaluates to: None
West (1,702) + 'not West' (848) = 2,550 of 3000 customers -> 450 fall out of both sides


Difference: WHERE region != 'West' returns 848 customers. WHERE region IS DISTINCT FROM 'West', the query the analyst meant, returns 1,298. The gap is exactly the 450 customers with no recorded region.

Mechanism: for a NULL region, != 'West' evaluates to NULL (unknown), not true, and WHERE keeps only true rows, those customers are dropped without warning.

Consequence: these customers also fail region = 'West', so they disappear from every regional filter: West (1,702) plus "not West" (848) covers only 2,550 of 3,000 customers. The averages barely move here (797.02 vs 796.26) only because the NULLs are random. If the missing customers are newer, smaller or more likely to churn. Because unknowns may cluster in some regions, comparisons between regions are biased by different amounts.

### Step 2.3: Aggregate without collapsing (10 pts)

In [109]:
# --- Step 2.3: aggregate without collapsing ---------------------
# GROUP BY region would collapse to one row per region. AVG (...) OVER (PARTITION BY region)
# computes the same averages but writes each one on every row of its region,
# so every customer keeps their own row and gains the comparison columns
WINDOW_SQL = """
    SELECT customer_id,
           region,
           revenue,
           AVG(revenue) OVER (PARTITION BY region)             AS region_avg,
           revenue - AVG(revenue) OVER (PARTITION BY region)   AS vs_region,
           CASE WHEN revenue > AVG(revenue) OVER (PARTITION BY region)
                THEN 'above' ELSE 'below' END                  AS position
FROM customers
ORDER BY customer_id
"""

with_avg = con.execute(WINDOW_SQL).df()

# Checks: one query, no join, no subquery, row count unchanged, averages correct
sql_up = WINDOW_SQL.upper()
assert " JOIN " not in sql_up and sql_up.count("SELECT") == 1, "one query: no JOIN and no subquery"
assert "OVER" in sql_up, "use a window function: AVG(...) OVER (PARTITION BY ...)"
assert len(with_avg) == N_CUSTOMERS, "a window function never changes the row count"
group_mean = with_avg.groupby("region", dropna=False)["revenue"].transform("mean")
assert np.allclose(with_avg["region_avg"], group_mean), "region_avg should be the average of each customer's own region"

print(with_avg.head(6).round(2).to_string(index=False))
print(f"\nrows in: {N_CUSTOMERS:,} | rows out: {len(with_avg):,}")
print(with_avg.groupby("region", dropna=False)
      .agg(customers=("customer_id", "size"),
           region_avg=("region_avg", "first"),
           above_avg=("position", lambda s: (s == "above").sum()))
      .round(2))

 customer_id  region  revenue  region_avg  vs_region position
           1    West   843.53      826.54      16.99    above
           2    West   546.37      826.54    -280.17    below
           3    West  2233.20      826.54    1406.66    above
           4    West  1171.16      826.54     344.62    above
           5    West   687.04      826.54    -139.50    below
           6 Central   467.37      803.89    -336.52    below

rows in: 3,000 | rows out: 3,000
         customers  region_avg  above_avg
region                                   
Central        284      803.89        105
East           284      764.82        108
South          280      822.72         99
West          1702      826.54        642
NaN            450      794.84        169


## Phase 3: The Causal Question (25 pts)

### Step 3.1: The programme evaluation (12 pts)

In [110]:
# --- Step 3.1: the programme evaluation -------
# Existing account size (the confounder) drives both who is in the West, where the
# programme ran, and revenue. The programme's true effect is small: +$25.

TRUE_EFFECT = 25.0
N_SIM = 5_000
account_size = rng.normal(100, 30, N_SIM)    # the confounder
programme = (rng.random(N_SIM) < 1 / (1 + np.exp(-(account_size - 100) / 15))).astype(int)  #bigger account -> West
revenue_sim = 300 + 5 * account_size + TRUE_EFFECT * programme + rng.normal(0, 50, N_SIM)
prog = pd.DataFrame({"account_size": account_size, "programme": programme, "revenue": revenue_sim})

naive = smf.ols("revenue ~ programme", data=prog).fit().params["programme"]
adjusted = smf.ols("revenue ~ programme + account_size", data=prog).fit().params["programme"]

print(f"truth {TRUE_EFFECT:+.2f} | naive {naive:+.2f} (bias {naive - TRUE_EFFECT:+.2f})")
print(f"truth {TRUE_EFFECT:+.2f} | adjusted {adjusted:+.2f} (bias {adjusted - TRUE_EFFECT:+.2f})")

assert naive > 3 * TRUE_EFFECT, "the confounder should inflate the naive gap well above the truth"
assert abs(adjusted - TRUE_EFFECT) < 15, "holding account size fixed should land close to the truth"

truth +25.00 | naive +213.72 (bias +188.72)
truth +25.00 | adjusted +26.88 (bias +1.88)


In this simulated world the programme's true effect is +25, but bigger existing accounts were more likely to be in the West, and account size raises revenue on its own. The naive West-versus-rest gap is +213.72, a bias of +188.72: almost all of it comes from West accounts being bigger to begin with. Holding account size fixed gives +26.88, close to the truth. West's higher revenue is mostly about who was in the West, not what the programme did, and the adjustment works only because we know the confounder and measured it.



### Step 3.2: The control that ruins it (13 pts)

In [111]:
# --- Step 3.2: the control that ruins it -------
# Support tickets are CAUSED BY both the programme (account managers get customers to log
# issues) and revenue (bigger spenders raise more tickets). Tickets do not cause revenue.
prog["tickets"] = rng.poisson(
    np.maximum(0, 2 + 4 * prog["programme"] + 0.02 * (prog["revenue"] - 800))
)

with_tickets = smf.ols("revenue ~ programme + account_size + tickets", data=prog).fit()
print(with_tickets.summary())

# Customers with the same ticket count: programme vs non-programme revenue
same = prog[prog["tickets"] == prog["tickets"].median()]
gap_same = same.groupby("programme")["revenue"].mean().diff().iloc[-1]

print(f"truth {TRUE_EFFECT:+.2f} | naive {naive:+.2f} (bias {naive - TRUE_EFFECT:+.2f})")
print(f"truth {TRUE_EFFECT:+.2f} | adjusted {adjusted:+.2f} (bias {adjusted - TRUE_EFFECT:+.2f})")
print(f"among customers with {prog['tickets'].median():.0f} tickets: programme minus rest = {gap_same:+.2f}")

assert with_tickets.params["programme"] < adjusted - 15, "controlling for tickets should pull the estimate away from the truth"

                            OLS Regression Results                            
Dep. Variable:                revenue   R-squared:                       0.919
Model:                            OLS   Adj. R-squared:                  0.919
Method:                 Least Squares   F-statistic:                 1.879e+04
Date:                Tue, 03 Nov 2026   Prob (F-statistic):               0.00
Time:                        23:30:00   Log-Likelihood:                -26452.
No. Observations:                5000   AIC:                         5.291e+04
Df Residuals:                    4996   BIC:                         5.294e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept      332.1444      2.856    116.283   

**Result:** holding account size fixed gives +26.88, close to the true +25. Adding support tickets as a control drops the estimate to **−1.06** (95% CI −5.18 to 3.07), so a programme that truly adds 25 now looks like it does nothing. Among customers with the same ticket count (4), programme customers bring in 24.79 less than the rest.

Support tickets sound like a measure of engagement, but in our data they go up for two separate reasons: the programme itself (account managers encourage customers to log issues), and spending (bigger customers raise more tickets). So when we compare customers with the same number of tickets, we are not comparing like with like. A programme customer reaches that ticket count partly because of the programme, so they need less spending to get there. A non-programme customer with the same count got there through spending alone, so they tend to be a bigger customer. "Same tickets" therefore quietly lines up smaller programme customers against bigger non-programme customers, and that wipes out the programme's real effect: it looks like it adds nothing, when it actually adds about 25 per customer. Tickets are a result of both the programme and revenue, not a cause of revenue, so holding them fixed hides a difference that is really there. The estimate to trust is the one that adjusts only for what existed before the programme started, existing account size, which puts the effect at about +27.

## Phase 4: AI Expansion — The Co-Pilot Era (20 pts)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

### Task 4.1: The query reviewer (12 pts)

In [112]:
import re
import pandas as pd

def review_query(con, sql):
    """
    Analyzes a SQL query for analytical pitfalls against DuckDB database context.
    Returns a list of dictionaries with keys: code, severity, message, evidence.
    """
    warnings = []
    sql_clean = re.sub(r'\s+', ' ', sql).strip()
    sql_upper = sql_clean.upper()

    try:
        # --- CTE Parsing ---
        ctes = []
        if "WITH" in sql_upper:
            # Grab names of CTEs
            cte_matches = re.findall(r'(\w+)\s+AS\s*\(', sql_clean, re.IGNORECASE)
            ctes = [c.upper() for c in cte_matches]

        # --- 1. Join Fan-Out and Uniqueness Check ---
        # Check if query contains any JOINs
        if "JOIN" in sql_upper:
            # Look for table names in joins
            # This basic parser handles LEFT JOIN / JOIN with optional aliases
            join_pat = r'(?:JOIN|LEFT\s+JOIN)\s+(\w+)(?:\s+(\w+))?\s+ON\s+([^\s]+)\s*=\s*([^\s]+)'
            joins = re.findall(join_pat, sql_clean, re.IGNORECASE)

            for table, alias, left_col, right_col in joins:
                table_upper = table.upper()
                if table_upper in ctes:
                    continue # Bypass CTEs (e.g. order_counts)

                # For the orders table, check if customer_id repeats
                if table_upper == "ORDERS":
                    join_col = "customer_id"
                    counts = con.execute("SELECT COUNT(*), COUNT(DISTINCT customer_id) FROM orders").fetchone()
                    total_rows, distinct_keys = counts[0], counts[1]

                    if total_rows > distinct_keys:
                        avg_rows = total_rows / distinct_keys
                        warnings.append({
                            "code": "FAN_OUT",
                            "severity": "MEDIUM",
                            "message": f"Table 'orders' is joined on non-unique key 'customer_id'. Each key repeats {avg_rows:.2f} times on average.",
                            "evidence": {
                                "query_rows": total_rows,
                                "query_distinct_left_keys": distinct_keys,
                                "average_rows_per_key": avg_rows
                            }
                        })

                        # 2. Check for Aggregates over Fan-out
                        # Find aggregates (AVG, COUNT, STDDEV_SAMP, SUM) excluding COUNT(DISTINCT ...)
                        agg_exprs = re.findall(r'\b(AVG|COUNT|SUM|STDDEV_SAMP)\s*\(\s*([^\)]+)\)', sql_clean, re.IGNORECASE)
                        affected_aggs = []
                        unaffected_aggs = []
                        for func, inside in agg_exprs:
                            expr = f"{func}({inside.strip()})"
                            if "DISTINCT" in inside.upper():
                                unaffected_aggs.append(expr)
                            else:
                                affected_aggs.append(expr)

                        if affected_aggs:
                            warnings.append({
                                "code": "AGG_OVER_FAN_OUT",
                                "severity": "HIGH",
                                "message": f"Aggregates {affected_aggs} computed over fanned-out rows will be weighted by repeats.",
                                "evidence": {
                                    "affected_aggregates": affected_aggs,
                                    "unaffected_aggregates": unaffected_aggs
                                }
                            })

        # --- 3. Null Filter Check (Three-Valued Logic) ---
        # Check if there is a WHERE clause
        if "WHERE" in sql_upper:
            where_part = sql_clean[sql_upper.find("WHERE") + 6:]
            # We target the 'region' column specifically in this context
            if "REGION" in where_part.upper():
                # Determine the comparison operator
                op_match = re.search(r'region\s*(=|!=|<>|\bIN\b|\bNOT\s+IN\b)', where_part, re.IGNORECASE)
                if op_match:
                    op = op_match.group(1).upper()
                    # Check if query already handles NULLs safely
                    has_null_handling = any(x in where_part.upper() for x in ["IS NULL", "IS DISTINCT FROM", "COALESCE"])
                    if not has_null_handling:
                        null_count = con.execute("SELECT COUNT(*) FROM customers WHERE region IS NULL").fetchone()[0]
                        if null_count > 0:
                            severity = "HIGH" if op in ["!=", "<>", "NOT IN"] else "INFO"
                            warnings.append({
                                "code": "NULL_FILTER",
                                "severity": severity,
                                "message": f"Filter using '{op}' on nullable column 'region' silently drops {null_count} unknown rows.",
                                "evidence": {
                                    "column": "region",
                                    "null_rows": null_count
                                }
                            })

    except Exception as e:
        warnings.append({
            "code": "CHECK_FAILED",
            "severity": "CRITICAL",
            "message": f"Unexpected parser error: {str(e)}",
            "evidence": {"error": str(e)}
        })

    return warnings

In [113]:
# Run review_query on your Phase 1 and Phase 2 queries.
# Two extra cases from the revised prompt: the older-SQL NULL fix and a safe join
OR_NULL_SQL = "SELECT COUNT(*) FROM customers WHERE region != 'West' OR region IS NULL"
SAFE_JOIN_SQL = """
    WITH order_counts AS (SELECT customer_id, COUNT(*) AS n_orders FROM orders GROUP BY customer_id)
    SELECT AVG(c.revenue) FROM customers c
    LEFT JOIN order_counts k ON k.customer_id = c.customer_id
    WHERE c.region = 'West'
"""

# What a correct reviewer should flag (INFO-level warnings are not counted as flags)
tests = {
    "LOW_SQL (Finance, Phase 1)":     (LOW_SQL,       set()),
    "HIGH_SQL (dashboard, Phase 1)":  (HIGH_SQL,      {"FAN_OUT", "AGG_OVER_FAN_OUT"}),
    "NE_SQL (!= 'West', Phase 2)":    (NE_SQL,        {"NULL_FILTER"}),
    "MEANT_SQL (IS DISTINCT FROM)":   (MEANT_SQL,     set()),
    "OR_NULL_SQL (!= OR IS NULL)":    (OR_NULL_SQL,   set()),
    "SAFE_JOIN_SQL (aggregate first)": (SAFE_JOIN_SQL, set()),
    "WINDOW_SQL (Step 2.3)":          (WINDOW_SQL,    set()),
}

rows = []
for name, (sql, expected) in tests.items():
    warnings = review_query(con, sql)
    # Default to an empty list if review_query returns None to avoid TypeError
    if warnings is None:
        warnings = []
    flagged = {w["code"] for w in warnings if w.get("severity") != "INFO"}
    rows.append({
        "query": name,
        "flagged": ", ".join(sorted(flagged)) or "-",
        "hits": ", ".join(sorted(expected & flagged)) or "-",
        "misses": ", ".join(sorted(expected - flagged)) or "-",
        "false alarms": ", ".join(sorted(flagged - expected)) or "-",
    })

results = pd.DataFrame(rows)
print(results.to_string(index=False))

                          query                   flagged                      hits misses false alarms
     LOW_SQL (Finance, Phase 1)                         -                         -      -            -
  HIGH_SQL (dashboard, Phase 1) AGG_OVER_FAN_OUT, FAN_OUT AGG_OVER_FAN_OUT, FAN_OUT      -            -
    NE_SQL (!= 'West', Phase 2)               NULL_FILTER               NULL_FILTER      -            -
   MEANT_SQL (IS DISTINCT FROM)                         -                         -      -            -
    OR_NULL_SQL (!= OR IS NULL)                         -                         -      -            -
SAFE_JOIN_SQL (aggregate first)                         -                         -      -            -
          WINDOW_SQL (Step 2.3)                         -                         -      -            -


Version 1: looked right but measured nothing.

Version 2: passed all seven test queries: 3 hits, 0 misses, 0 false alarms.

### Task 4.2: The one-page writeup (8 pts)

To the director:

West revenue per customer and the account-management programme. All figures are in dollars.

Which number is right: Finance's 812 is the correct average revenue per customer. In our rebuild, the um-joined query gives 823.93 per West customer and the dashboard's query gives 1,256.76, reproducing the gap.

Why they differed: The dashboard joins customers to orders before averaging, so each customer is counted once per order: 2,000 West customers becomes 14,004 rows. Bigger customers place more orders, so they are counted more often and the average is inflated. Separetely, 15% of customers have no recorded region, and standard filters drop them every regional comparison without warning.

Does the programme? The West-versus-rest gap cannot tell us. In a simulation where the true effect is +25, the raw gap was +213.72, almost all of it because West accounts were bigger to begin with. Adjusting for pre-programe account size recovered +26.88. Adding support tickets as a control, which the programme itself raises, erased the effect (-1.06). On real data, the effect is unknown until we compare like with like.

### AI Transcript

Paste **your prompt exactly as you sent it** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:* Opus 5.5 High / 11.3.2026

*Prompt:* [Prep] Act as an analytics engineer who reviews SQL before it reaches
a dashboard.

[Request] Write a Python function review_query(con, sql) that runs
against DuckDB and returns a list of warnings for a given SELECT. It
should flag at minimum: a join that changes the row count, a filter on
a nullable column that will silently drop unknowns, and an aggregate
computed over a fanned-out result.

[Iterate] duckdb and pandas only. Reuse my table names, customers and
orders. Return structured warnings, not printed text, so I can test it.

[Mechanism Check] Add comments explaining how you detect fan-out
without running the full join twice, and why three-valued logic makes
the null filter a correctness issue rather than a style issue.

[Evaluate] Run it against both of my Phase 1 queries and tell me
whether it would have caught the $1,240 before it reached the board.

*Reply:*

from review_query import review_query, warnings_frame
FINANCE = "SELECT AVG(revenue) FROM customers WHERE region = 'West'"
DASHBOARD = """SELECT AVG(c.revenue) FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id WHERE c.region = 'West'"""
for name, q in [("Finance", FINANCE), ("Dashboard", DASHBOARD)]:
    print(name); display(warnings_frame(review_query(con, q))[["code", "severity", "message"]])

------------


*Model and date:* Opus 5.5 High / 11.3.2026

*Prompt:*

[Prep] Act as an analytics engineer who reviews SQL before it reaches a dashboard. Your first version has problems I need fixed: the fan-out check computes base_count but never uses it, so it flags any query with JOIN and AVG( instead of measuring duplicates; the null check is hard-coded to region and never checks whether the column actually contains NULLs; it flags region = 'West' and "region != 'West' OR region IS NULL" as errors; and except Exception: pass hides failures.

[Request] Rewrite review_query(con, sql) so it returns a list of warning dicts with keys code, severity, message and evidence (the counts behind the warning). It must:
1. FAN_OUT: for each table joined in the query, check whether the join key is unique on that table (COUNT(*) vs COUNT(DISTINCT key)). Warn only when the key repeats, and report how many rows per key on average. Do not warn when the join is to a CTE or table that already has one row per key.
2. AGG_OVER_FAN_OUT: warn when an aggregate (AVG, SUM, COUNT) is computed over a join that fans out, and say the result is weighted by the repeated rows.
3. NULL_FILTER: for every column used in a WHERE comparison (=, !=, <>, <, >, IN, NOT IN), query the table to count NULLs in that column. If it has NULLs and the query does not handle them (IS NULL, IS DISTINCT FROM, COALESCE), warn and report how many rows the filter drops. Use severity HIGH for !=, <> and NOT IN (they drop rows the analyst meant to keep) and INFO for = and IN (unknowns cannot be confirmed as the value).
4. Handle table aliases (c.region) and WITH clauses. No bare except: if a check fails, return a warning with code CHECK_FAILED and the error.

[Iterate] duckdb and pandas only. Tables: customers (customer_id, region, revenue, signup_date; region has NULLs) and orders (order_id, customer_id). Test it on these exact queries:

LOW_SQL:
SELECT AVG(revenue) AS avg_revenue, COUNT(*) AS n_rows, COUNT(DISTINCT customer_id) AS n_customers, STDDEV_SAMP(revenue) / SQRT(COUNT(*)) AS se FROM customers WHERE region = 'West'

HIGH_SQL:
SELECT AVG(c.revenue) AS avg_revenue, COUNT(*) AS n_rows, COUNT(DISTINCT c.customer_id) AS n_customers, STDDEV_SAMP(c.revenue) / SQRT(COUNT(*)) AS se FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id WHERE c.region = 'West'

NE_SQL:
SELECT COUNT(*), AVG(revenue) FROM customers WHERE region != 'West'

MEANT_SQL:
SELECT COUNT(*), AVG(revenue) FROM customers WHERE region IS DISTINCT FROM 'West'

OR_NULL_SQL:
SELECT COUNT(*) FROM customers WHERE region != 'West' OR region IS NULL

SAFE_JOIN_SQL:
WITH order_counts AS (SELECT customer_id, COUNT(*) AS n_orders FROM orders GROUP BY customer_id) SELECT AVG(c.revenue) FROM customers c LEFT JOIN order_counts k ON k.customer_id = c.customer_id WHERE c.region = 'West'

Expected: HIGH_SQL -> FAN_OUT and AGG_OVER_FAN_OUT; NE_SQL -> NULL_FILTER (HIGH); LOW_SQL, HIGH_SQL and SAFE_JOIN_SQL -> at most NULL_FILTER (INFO); MEANT_SQL and OR_NULL_SQL -> no warnings; SAFE_JOIN_SQL -> no fan-out warning.

[Mechanism Check] Add comments explaining how you detect fan-out without running the full join twice, and why three-valued logic makes the null filter a correctness issue rather than a style issue.

[Evaluate] Give me a test cell that runs review_query on all six queries and asserts the expected warnings above. Then tell me whether it would have caught the $1,240 dashboard number by measuring duplicate rows, not just by matching the words JOIN and AVG, and list any cases your function still cannot catch.

*Reply:*
(pasted at the top)

What I got. On the seed-42 data with Step 2.1’s NULLs, the reviewer caught 3 of 3 expected problems, with 0 misses, 0 false alarms, and 7 of 7 queries passing. I tested in SQLite again, and I couldn’t see your exact WINDOW_SQL, so I used a window AVG(revenue) OVER (PARTITION BY region) over all customers. If yours also filters on region = 'West', it will only add an INFO, which doesn’t count.

Checking the scorer itself. I also scored a word-matching reviewer that flags any JOIN with AVG(, and any !=. It also caught 3 of 3, but it raised 3 false alarms and failed exactly on OR_NULL_SQL and SAFE_JOIN_SQL. So the table can tell a tool that measures from one that matches words, which a broken-only test couldn’t do.

How to read the table. Each row is one query.

hits are real problems found.
misses are real problems the tool let through.
false_alarms are flags with no problem behind them.
info_not_counted shows the INFO notes, which are visible but not scored. The INFO on LOW_SQL is the 450 unknown regions that = 'West' can’t confirm; it’s not a false alarm.
A non-empty check_failed column means the reviewer couldn’t review that query, so treat it as a miss until it’s fixed.

What a miss costs. A miss on HIGH_SQL means the $1,240 passes review and reaches the board deck, which is exactly what happened in Phase 1. AGG_OVER_FAN_OUT is the miss that matters most. FAN_OUT alone (MEDIUM) says the grain changed, but only AGG_OVER_FAN_OUT says the average itself is weighted by order count, and wrong by about 50%.

What a false alarm costs. First, the analyst’s time: they investigate a correct query, or worse, “fix” it and change a right number. Second, and more expensive, trust. After a few warnings like the ones on SAFE_JOIN_SQL, analysts learn to dismiss FAN_OUT. Then the one real fan-out, the next $1,240, gets dismissed too. A reviewer with many false alarms ends up catching as little as one with many misses, because no one acts on its warnings.

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; your prompt is in it as sent.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 3/Econ_3916_PS3.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.